In [5]:

import requests

# 1. Show the raw API response structure (JSON)
url = 'https://power.larc.nasa.gov/api/temporal/daily/point'
params = {
    'parameters': 'T2M,RH2M,PRECTOTCORR,WS10M',
    'community': 'RE',
    'longitude': 74.3587,
    'latitude': 31.5204,
    'start': '20230101',
    'end': '20230107',
    'format': 'JSON'
}
response = requests.get(url, params=params, timeout=30)
data = response.json()

print("Raw JSON keys:", list(data.keys()))
print("\nProperties keys:", list(data['properties'].keys()))
print("\nSample of T2M data:")
print(data['properties']['parameter']['T2M'])

ModuleNotFoundError: No module named 'requests'

In [ ]:
import pandas as pd
import numpy as np

# 2. The daily DataFrame (.head())
paramsData = data['properties']['parameter']
dfDaily = pd.DataFrame(paramsData)

# IMMEDIATELY replace -999.0 with np.nan
dfDaily = dfDaily.replace(-999.0, np.nan)

dfDaily.index = pd.to_datetime(dfDaily.index, format='%Y%m%d')
dfDaily.index.name = 'date'
dfDaily.head()

,T2M,RH2M,PRECTOTCORR,WS10M
date,,,,
2023-01-01,11.95,51.44,0.0,1.64
2023-01-02,12.25,40.31,0.0,1.48
2023-01-03,11.41,39.18,0.0,2.17
2023-01-04,10.18,43.61,0.0,2.52
2023-01-05,10.64,37.98,0.0,2.31


In [1]:
import sys
sys.path.append('..')
from backend.utils.nasa_fetcher import fetch_weather
import pandas as pd

# Punjab districts only — 36 coordinates
districtCoords = {
    "Lahore":          (31.5204, 74.3587),
    "Faisalabad":      (31.4504, 73.1350),
    "Rawalpindi":      (33.5651, 73.0169),
    "Gujranwala":      (32.1877, 74.1945),
    "Multan":          (30.1575, 71.5249),
    "Sheikhupura":     (31.7167, 73.9850),
    "Kasur":           (31.1167, 74.4500),
    "Sialkot":         (32.4945, 74.5229),
    "Sargodha":        (32.0836, 72.6711),
    "Bahawalpur":      (29.3956, 71.6836),
    "Jhang":           (31.2681, 72.3181),
    "Gujrat":          (32.5736, 74.0790),
    "Rahim Yar Khan":  (28.4202, 70.2952),
    "Sahiwal":         (30.6706, 73.1064),
    "Okara":           (30.8138, 73.4534),
    "Narowal":         (32.1000, 74.8700),
    "Hafizabad":       (32.0714, 73.6883),
    "Mandi Bahauddin": (32.5864, 73.4917),
    "Pakpattan":       (30.3436, 73.3869),
    "Chiniot":         (31.7200, 72.9800),
    "Khushab":         (32.2969, 72.3527),
    "Bhakkar":         (31.6278, 71.0642),
    "Layyah":          (30.9614, 70.9394),
    "Muzaffargarh":    (30.0736, 71.1936),
    "Lodhran":         (29.5333, 71.6333),
    "Vehari":          (30.0453, 72.3519),
    "Khanewal":        (30.3017, 71.9322),
    "Toba Tek Singh":  (30.9700, 72.4800),
    "Nankana Sahib":   (31.4500, 73.7100),
    "Attock":          (33.7664, 72.3601),
    "Chakwal":         (32.9328, 72.8571),
    "Jhelum":          (32.9350, 73.7203),
    "Mianwali":        (32.5836, 71.5433),
    "D.G. Khan":       (30.0500, 70.6333),
    "Rajanpur":        (29.1042, 70.3300),
    "Bahawalnagar":    (29.9978, 73.2536),
}

# fetch NASA weather per district — 36 API calls,
weatherFrames = []
failed = []

for district, (lat, lon) in districtCoords.items():
    print(f"Fetching {district}...")
    try:
        df_wx = fetch_weather(lat, lon, "20160101", "20201231")
        df_wx["district"] = district
        weatherFrames.append(df_wx)
    except Exception as e:
        print(f"  FAILED {district}: {e}")
        failed.append(district)

if failed:
    print(f"\nFailed districts — re-run manually: {failed}")
else:
    print("\nAll 36 districts fetched successfully")

dfWeather = pd.concat(weatherFrames, ignore_index=True)
print(f"Weather shape: {dfWeather.shape}")

Fetching Lahore...
Fetching Faisalabad...
Fetching Rawalpindi...
Fetching Gujranwala...
Fetching Multan...
Fetching Sheikhupura...
Fetching Kasur...
Fetching Sialkot...
Fetching Sargodha...
Fetching Bahawalpur...
Fetching Jhang...
Fetching Gujrat...
Fetching Rahim Yar Khan...
Fetching Sahiwal...
Fetching Okara...
Fetching Narowal...
Fetching Hafizabad...
Fetching Mandi Bahauddin...
Fetching Pakpattan...
Fetching Chiniot...
Fetching Khushab...
Fetching Bhakkar...
Fetching Layyah...
Fetching Muzaffargarh...
Fetching Lodhran...
Fetching Vehari...
Fetching Khanewal...
Fetching Toba Tek Singh...
Fetching Nankana Sahib...
Fetching Attock...
Fetching Chakwal...
Fetching Jhelum...
Fetching Mianwali...
Fetching D.G. Khan...
Fetching Rajanpur...
Fetching Bahawalnagar...

All 36 districts fetched successfully
Weather shape: (9432, 7)


In [ ]:
import pandas as pd


def convertToCSV(file, output_file=None, **kwargs):
    df = pd.read_excel(file, **kwargs)
    
    if output_file is None:
        output_file = file.rsplit('.', 1)[0] + '.csv'
    
    df.to_csv(
        output_file,
        index=False,
        encoding='utf-8-sig'
    )
    
    print(f"Converted: {file} → {output_file}")
    return output_file



In [2]:
import pandas as pd
import numpy as np


# Realistic week-of-month patterns for off-season months
# Even "normal" months have slight variation due to reporting delays, 
# partial weeks at month boundaries, etc.
offseasonPatterns = [
    [0.22, 0.28, 0.28, 0.22],  # slight middle bump (common)
    [0.24, 0.26, 0.26, 0.24],  # nearly flat (stable period)
    [0.20, 0.30, 0.30, 0.20],  # stronger middle (minor outbreak)
    [0.23, 0.27, 0.27, 0.23],  # gentle slope up then down
]

# Outbreak months: strong mid-month spike (weeks 2-3)
# These get small random perturbation for year-to-year realism
outbreakBase = [0.15, 0.35, 0.35, 0.15]

# True epidemiological week mapping (52-week year)
# Each month gets ~4.3 weeks; we distribute to avoid gaps

monthNums = {
    "January":1, "February":2, "March":3, "April":4,
    "May":5, "June":6, "July":7, "August":8,
    "September":9, "October":10, "November":11, "December":12
}

monthToStartWeek = {
   "January":1, "February":5, "March":9, "April":13,
    "May":18, "June":22, "July":26, "August":31,
    "September":35, "October":40, "November":44, "December":48
}

df=pd.read_csv('../data/raw/dengue_data.csv')

In [3]:
punjabDistrictsWeights = pd.DataFrame([
    ["Lahore",          11126285, 0.95, 0.60],
    ["Faisalabad",       7873910, 0.80, 0.40],
    ["Rawalpindi",       5405633, 0.70, 0.50],
    ["Gujranwala",       5014978, 0.75, 0.60],
    ["Multan",           4745209, 0.60, 0.50],
    ["Sheikhupura",      3321029, 0.70, 0.75],
    ["Kasur",            2981000, 0.65, 0.80],
    ["Sialkot",          3893000, 0.60, 0.55],
    ["Sargodha",         3385000, 0.50, 0.45],
    ["Bahawalpur",       3668000, 0.35, 0.40],
    ["Jhang",            2834000, 0.55, 0.70],
    ["Gujrat",           2446000, 0.55, 0.50],
    ["Rahim Yar Khan",   4814000, 0.30, 0.45],
    ["Sahiwal",          2207000, 0.45, 0.40],
    ["Okara",            2797000, 0.40, 0.45],
    ["Narowal",          1453000, 0.50, 0.55],
    ["Hafizabad",        1136000, 0.50, 0.60],
    ["Mandi Bahauddin",  1192000, 0.45, 0.55],
    ["Pakpattan",        1529000, 0.35, 0.40],
    ["Chiniot",          1264000, 0.50, 0.50],
    ["Khushab",          1132000, 0.35, 0.35],
    ["Bhakkar",          1491000, 0.30, 0.35],
    ["Layyah",           1722000, 0.30, 0.40],
    ["Muzaffargarh",     3708000, 0.45, 0.65],
    ["Lodhran",          1715000, 0.35, 0.40],
    ["Vehari",           2895000, 0.40, 0.40],
    ["Khanewal",         2765000, 0.40, 0.45],
    ["Toba Tek Singh",   2191000, 0.45, 0.50],
    ["Nankana Sahib",    1461000, 0.55, 0.55],
    ["Attock",           1575000, 0.40, 0.45],
    ["Chakwal",          1083000, 0.35, 0.30],
    ["Jhelum",           1103000, 0.40, 0.50],
    ["Mianwali",         1560000, 0.30, 0.40],
    ["D.G. Khan",        2982000, 0.35, 0.55],
    ["Rajanpur",         1994000, 0.30, 0.60],
    ["Bahawalnagar",     2978000, 0.30, 0.35],
],
columns=["district", "population", "dengue_history", "flood_risk"])

# compute combined risk weight
punjabDistrictsWeights["pop_w"]     = (
    punjabDistrictsWeights["population"] /
    punjabDistrictsWeights["population"].sum()
)
punjabDistrictsWeights["hist_w"]    = (
    punjabDistrictsWeights["dengue_history"] /
    punjabDistrictsWeights["dengue_history"].sum()
)
punjabDistrictsWeights["flood_w"]   = (
    punjabDistrictsWeights["flood_risk"] /
    punjabDistrictsWeights["flood_risk"].sum()
)
punjabDistrictsWeights["risk_weight"] = (
    punjabDistrictsWeights["pop_w"]   * 0.50 +
    punjabDistrictsWeights["hist_w"]  * 0.30 +
    punjabDistrictsWeights["flood_w"] * 0.20
)
punjabDistrictsWeights["risk_weight"] = (
    punjabDistrictsWeights["risk_weight"] /
    punjabDistrictsWeights["risk_weight"].sum()
)

print(punjabDistrictsWeights[["district","risk_weight"]]
      .sort_values("risk_weight", ascending=False).to_string())

           district  risk_weight
0            Lahore     0.076150
1        Faisalabad     0.055855
3        Gujranwala     0.043662
2        Rawalpindi     0.043516
4            Multan     0.038630
5       Sheikhupura     0.036432
7           Sialkot     0.035149
6             Kasur     0.034503
12   Rahim Yar Khan     0.033133
23     Muzaffargarh     0.032761
10            Jhang     0.030931
8          Sargodha     0.029866
9        Bahawalpur     0.028016
11           Gujrat     0.026850
33        D.G. Khan     0.026443
14            Okara     0.025323
25           Vehari     0.025227
26         Khanewal     0.025171
27   Toba Tek Singh     0.023887
35     Bahawalnagar     0.023306
13          Sahiwal     0.022842
28    Nankana Sahib     0.022740
15          Narowal     0.021825
34         Rajanpur     0.021441
16        Hafizabad     0.020882
19          Chiniot     0.020368
17  Mandi Bahauddin     0.019710
29           Attock     0.019528
24          Lodhran     0.018755
22        

In [4]:
def allocate_exact(total, weights):
    """
    Distribute total across bins by weights.
    Guarantees sum(result) == total exactly.
    """
    if total == 0:
        return [0] * len(weights)

    raw       = [total * w for w in weights]
    allocated = [int(x) for x in raw]          # floor, not round
    remainder = total - sum(allocated)          # always >= 0 when using floor

    # give remainder to largest fractional parts
    fractions = sorted(
        range(len(raw)),
        key=lambda i: raw[i] - allocated[i],
        reverse=True
    )
    for i in range(remainder):
        allocated[fractions[i % len(fractions)]] += 1

    assert sum(allocated) == total, f"allocate_exact failed: {sum(allocated)} != {total}"
    return allocated

In [5]:
dfWeekly=[]


for _, row in df.iterrows():
    month     = str(row["Month"])
    month_num = monthNums[month]
    year      = int(row["Year"])
    region    = row["Region"]

    if region != "Punjab":
        continue

    total_cases  = int(row["Dengue_Cases"])
    total_deaths = int(row["Dengue_Deaths"])
    start_week   = monthToStartWeek[month]

    carry_cases  = 0.0
    carry_deaths = 0.0
    districtCaseTotals = []

    for i, (_, dist_row) in enumerate(punjabDistrictsWeights.iterrows()):
        district    = dist_row["district"]
        dist_weight = dist_row["risk_weight"]

        # exact float share + carry from previous district
        raw_cases  = total_cases  * dist_weight + carry_cases
        raw_deaths = total_deaths * dist_weight + carry_deaths

        # round to integer — store what we couldn't fit
        dist_cases  = round(raw_cases)
        dist_deaths = round(raw_deaths)

        carry_cases  = raw_cases  - dist_cases
        carry_deaths = raw_deaths - dist_deaths

        # last district — flush any remaining carry
        if i == len(punjabDistrictsWeights) - 1:
            dist_cases  += int(carry_cases)
            dist_deaths += int(carry_deaths)
            carry_cases  = 0.0
            carry_deaths = 0.0

        districtCaseTotals.append(dist_cases)

        # temporal weights
        if month_num in [7, 8, 9, 10]:
            rng = np.random.default_rng(
                hash(f"{year}{month_num}{district}") % (2**32)
            )
            noise   = rng.normal(0, 0.02, 4)
            weights = [max(0.05, w + n) for w, n in zip(outbreakBase, noise)]
        else:
            import hashlib
            region_hash = int(hashlib.md5(district.encode()).hexdigest(), 16)
            pattern_idx = (year + month_num + region_hash) % len(offseasonPatterns)
            weights     = list(offseasonPatterns[pattern_idx])

        weights = [w / sum(weights) for w in weights]

        week_cases  = allocate_exact(dist_cases,  weights)
        week_deaths = allocate_exact(dist_deaths, weights)

        for w in range(4):
            epi_week = min(start_week + w, 52)
            dfWeekly.append({
                "year":      year,
                "month":     month,
                "month_num": month_num,
                "epi_week":  epi_week,
                "region":    region,
                "district":  district,
                "cases":     week_cases[w],
                "deaths":    week_deaths[w],
            })

dfWeekly = pd.DataFrame(dfWeekly)

In [6]:
# Verify: sum of all districts should equal original province total
weeklyTotals = dfWeekly.groupby(["year", "month"])["cases"].sum()
originalTotals = df[df["Region"] == "Punjab"].groupby(["Year", "Month"])["Dengue_Cases"].sum()

print("Verification (should match):")
for (year, month), orig in originalTotals.items():
    weekly = weeklyTotals.get((year, month), 0)
    if abs(orig - weekly) > 10:
        print(f"  MISMATCH {year}-{month}: orig={orig}, weekly_sum={weekly}")

print(f"\nDistricts: {dfWeekly['district'].nunique()}")
print(f"Sample Lahore 2019:")
print(dfWeekly[
    (dfWeekly["district"] == "Lahore") & 
    (dfWeekly["year"] == 2019) & 
    (dfWeekly["epi_week"].between(31, 40))
].to_string())

dfWeekly.to_csv("../data/processed/punjab_district_weekly.csv", index=False)

Verification (should match):

Districts: 36
Sample Lahore 2019:
      year      month  month_num  epi_week  region district  cases  deaths
6192  2019     August          8        31  Punjab   Lahore    139       1
6193  2019     August          8        32  Punjab   Lahore    356       1
6194  2019     August          8        33  Punjab   Lahore    326       1
6195  2019     August          8        34  Punjab   Lahore    127       1
6336  2019  September          9        35  Punjab   Lahore    111       0
6337  2019  September          9        36  Punjab   Lahore    222       1
6338  2019  September          9        37  Punjab   Lahore    228       1
6339  2019  September          9        38  Punjab   Lahore    103       0
6480  2019    October         10        40  Punjab   Lahore     44       0


In [13]:

data=pd.read_csv('../data/processed/updated-Punjab.csv')

print("The shape of the final Engineered dataset is ",data.shape)

statsSummary=data.describe()
print("The statistical Summmary of the Final Engineered dataset is ")
print(statsSummary)

print("================================================================================")

The shape of the final Engineered dataset is  (8568, 21)
The statistical Summmary of the Final Engineered dataset is 
              year    month_num     epi_week        cases       deaths  \
count  8568.000000  8568.000000  8568.000000  8568.000000  8568.000000   
mean   2018.016807     6.546218    26.289916     6.617997     0.012838   
std       1.408241     3.429554    14.892494    16.148621     0.112584   
min    2016.000000     1.000000     1.000000     0.000000     0.000000   
25%    2017.000000     4.000000    14.000000     2.000000     0.000000   
50%    2018.000000     7.000000    27.000000     3.000000     0.000000   
75%    2019.000000    10.000000    40.000000     5.000000     0.000000   
max    2020.000000    12.000000    51.000000   362.000000     1.000000   

          T2M_mean    RH2M_mean  PRECTOTCORR_sum   WS10M_mean     month_sin  \
count  8568.000000  8568.000000      8568.000000  8568.000000  8.568000e+03   
mean     25.945487    41.392683        11.722096     2.51

In [14]:
data=pd.read_csv('../data/raw/dengue_data.csv')

print("The shape of the primary dataset is ",data.shape)

statsSummary=data.describe()
print("The statistical Summmary of the primary dataset is ")
print(statsSummary)

print("================================================================================")

The shape of the primary dataset is  (420, 5)
The statistical Summmary of the primary dataset is 
            Year  Dengue_Cases  Dengue_Deaths
count   420.0000    420.000000     420.000000
mean   2018.0000    421.750000       2.773810
std       1.4159    992.468102       4.605567
min    2016.0000     15.000000       0.000000
25%    2017.0000     80.000000       0.000000
50%    2018.0000    180.000000       1.000000
75%    2019.0000    350.000000       3.250000
max    2020.0000  12450.000000      48.000000


In [15]:
data=pd.read_csv('../data/raw/nasa_power_raw.csv')

print("The shape of the NASA Power dataset is ",data.shape)

statsSummary=data.describe()
print("The statistical Summmary of the NASA Power dataset is ")
print(statsSummary)

print("================================================================================")

The shape of the NASA Power dataset is  (262, 6)
The statistical Summmary of the NASA Power dataset is 
              year    epi_week    T2M_mean   RH2M_mean  PRECTOTCORR_sum  \
count   262.000000  262.000000  262.000000  262.000000       262.000000   
mean   2017.996183   26.702290   25.345438   44.628438        13.259046   
std       1.429031   15.156752    8.419064   15.519970        20.916952   
min    2015.000000    1.000000    8.670000   12.605714         0.000000   
25%    2017.000000   14.000000   17.270000   31.978571         0.020000   
50%    2018.000000   27.000000   27.130000   45.439286         2.980000   
75%    2019.000000   40.000000   32.574643   56.371786        18.430000   
max    2020.000000   53.000000   38.840000   82.794286       133.620000   

       WS10M_mean  
count  262.000000  
mean     2.370813  
std      0.527407  
min      1.271429  
25%      1.971786  
50%      2.367143  
75%      2.678929  
max      4.002857  


In [7]:

dengue=pd.read_csv('../data/processed/punjab_district_weekly.csv')

# 3. Check for mismatches before merging
print("\n=== PRE-MERGE CHECKS ===")

# District names
dengueDistricts = set(dengue['district'].unique())
weatherDistricts = set(dfWeather['district'].unique())

print(f"Dengue districts: {len(dengueDistricts)}")
print(f"Weather districts: {len(weatherDistricts)}")
print(f"Missing in weather: {dengueDistricts - weatherDistricts}")
print(f"Missing in dengue: {weatherDistricts - dengueDistricts}")

# Year ranges
print(f"\nDengue years: {sorted(dengue['year'].unique())}")
print(f"Weather years: {sorted(dfWeather['year'].unique())}")

# Week ranges
print(f"\nDengue weeks: {dengue['epi_week'].min()}-{dengue['epi_week'].max()}")
print(f"Weather weeks: {dfWeather['epi_week'].min()}-{dfWeather['epi_week'].max()}")

# 4. MERGE
merged = pd.merge(
    dengue,
    dfWeather,
    on=['year', 'epi_week', 'district'],
    how='left',
    indicator=True
)
merged = merged.drop(columns=['_merge'])
merged.to_csv('../data/processed/punjab_district_weekly_merged.csv', index=False)


=== PRE-MERGE CHECKS ===
Dengue districts: 36
Weather districts: 36
Missing in weather: set()
Missing in dengue: set()

Dengue years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020)]
Weather years: [np.uint32(2015), np.uint32(2016), np.uint32(2017), np.uint32(2018), np.uint32(2019), np.uint32(2020)]

Dengue weeks: 1-51
Weather weeks: 1-53


In [40]:
print(dfWeather)

      year  epi_week   T2M_mean  RH2M_mean  PRECTOTCORR_sum  WS10M_mean  \
0     2015        53  15.933333  25.073333             0.00    2.263333   
1     2016         1  15.575714  40.798571             0.05    2.288571   
2     2016         2  14.078571  46.415714             6.53    2.097143   
3     2016         3  12.247143  31.330000             0.00    1.967143   
4     2016         4  14.121429  46.512857            12.92    2.284286   
...    ...       ...        ...        ...              ...         ...   
9427  2020        49  17.232857  52.850000             0.00    1.707143   
9428  2020        50  15.505714  69.308571             7.29    2.381429   
9429  2020        51  12.591429  48.468571             0.00    1.587143   
9430  2020        52  13.998571  48.504286             0.00    1.770000   
9431  2020        53  10.687500  57.267500             0.08    1.807500   

          district  
0           Lahore  
1           Lahore  
2           Lahore  
3           Lah